## Doing Lab for chapter 02

### Get Titanic dataset

In [1]:
import pandas as pd
df_Titanic = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/Titanic_train.csv')
print('Titanic dataset')
display(df_Titanic.head(10))
display(df_Titanic.info())

Titanic dataset


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


None

### Preprocessing data

In [ ]:
df_Titanic_Cleaned = df_Titanic.drop(['PassengerId','Name','Ticket','Cabin'],axis=1)
df_Titanic_Cleaned['Age'] = df_Titanic_Cleaned['Age'].fillna(df_Titanic_Cleaned['Age'].mean) #fillna bang GT TB
df_Titanic_Cleaned['Embarked'] = df_Titanic_Cleaned['Embarked'].dropna()

### Mã hóa dữ liệu (Encoding) 

### Lab trên lớp Moudle 8

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Nạp dữ liệu
df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/[Dataset]_Module8_(Viral).csv')

# Lưu ý: Tên cột trong file này có khoảng trắng ở đầu, cần chuẩn hóa
df.columns = df.columns.str.strip()

# 2. Chọn đặc trưng và biến mục tiêu
# Loại bỏ cột 'url' (không có ý nghĩa dự đoán), cột 'shares' (rò rỉ dữ liệu) và cột 'Popular'
X = df.drop(columns=['url', 'shares', 'Popular'])
y = df['Popular']

# 3. Chia tập Train/Test theo tỷ lệ 80/20 giống Orange
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Huấn luyện mô hình
# Cây quyết định (Tree)
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

# Rừng ngẫu nhiên (Random Forest)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

# 5. Đánh giá mô hình trên tập Test
models = {'Decision Tree': tree_model, 'Random Forest': rf_model}

for name, model in models.items():
    print(f"\n--- Kết quả đánh giá mô hình {name} ---")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Xác suất cho lớp 1
    
    print(classification_report(y_test, y_pred))
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    
    # Hiển thị Confusion Matrix (tùy chọn)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('Thực tế')
    plt.xlabel('Dự đoán')
    plt.show()

### Phân loại Spam/Ham bằng Native Bayes

In [ ]:
import pandas as pd
import time
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, f1_score, roc_curve, auc)
from sklearn.pipeline import Pipeline

# =========================================================
# 1. NẠP VÀ LÀM SẠCH DỮ LIỆU
# =========================================================
df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/all_emails.csv')

df['text'] = df['text'].fillna('')

print("=" * 60)
print("PHÂN BỐ DỮ LIỆU (0 = Ham, 1 = Spam):")
print(df['spam'].value_counts())
print(f"Tỷ lệ Spam: {df['spam'].mean() * 100:.2f}%")
print("=" * 60)

X = df['text']
y = df['spam']

# =========================================================
# 2. CHIA TẬP TRAIN/TEST (Tỷ lệ 80-20)
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 3. ĐỊNH NGHĨA CÁC MÔ HÌNH CẦN SO SÁNH
# =========================================================
def create_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english',
            max_features=5000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ('clf', classifier)
    ])

models = {
    'Naive Bayes': create_pipeline(MultinomialNB()),
    'Logistic Regression': create_pipeline(
        LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    ),
    'Linear SVC': create_pipeline(
        LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
    )
}

# =========================================================
# 4. HUẤN LUYỆN VÀ ĐÁNH GIÁ TỪNG MÔ HÌNH
# =========================================================
results = {}

print("\n" + "=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ CÁC MÔ HÌNH")
print("=" * 60)

for name, pipeline in models.items():
    print(f"\n>>> Đang huấn luyện: {name} ...")
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # --- LƯU Ý QUAN TRỌNG: Lấy điểm số để vẽ ROC ---
    # Naive Bayes và Logistic Regression có predict_proba (xác suất)
    # Linear SVC không có predict_proba, phải dùng decision_function (khoảng cách)
    if hasattr(pipeline, 'predict_proba'):
        y_score = pipeline.predict_proba(X_test)[:, 1]
    elif hasattr(pipeline, 'decision_function'):
        y_score = pipeline.decision_function(X_test)
    else:
        raise AttributeError(f"Mô hình {name} không hỗ trợ tính toán xác suất hoặc decision function.")
    
    results[name] = {
        'model': pipeline,
        'accuracy': acc,
        'f1_score': f1,
        'train_time': train_time,
        'y_pred': y_pred,
        'y_score': y_score  # Lưu lại để vẽ ROC
    }
    
    print(f"Thời gian huấn luyện: {train_time:.4f} giây")
    print(f"Accuracy: {acc:.4f} | F1-Score (Spam): {f1:.4f}")
    print("-" * 60)
    print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

# =========================================================
# 5. BẢNG SO SÁNH TỔNG HỢP
# =========================================================
print("\n" + "=" * 60)
print("BẢNG SO SÁNH TỔNG HỢP CÁC MÔ HÌNH")
print("=" * 60)

comparison_df = pd.DataFrame({
    name: {
        'Accuracy': res['accuracy'],
        'F1-Score (Spam)': res['f1_score'],
        'Thời gian (s)': res['train_time']
    }
    for name, res in results.items()
}).T

print(comparison_df.round(4))

# =========================================================
# 6. VẼ MA TRẬN NHẦM LẪN CHO TẤT CẢ MÔ HÌNH
# =========================================================
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ham (0)', 'Spam (1)'],
                yticklabels=['Ham (0)', 'Spam (1)'])
    ax.set_title(f'Ma trận nhầm lẫn\n{name}')
    ax.set_ylabel('Thực tế')
    ax.set_xlabel('Dự đoán')

plt.tight_layout()
plt.show()

# =========================================================
# 7. VẼ BIỂU ĐỒ ROC CURVE SO SÁNH CÁC MÔ HÌNH (MỚI THÊM)
# =========================================================
plt.figure(figsize=(9, 7))

# Bảng màu đẹp cho 3 mô hình
colors = {
    'Naive Bayes': '#3498DB',       # Xanh dương
    'Logistic Regression': '#1ABC9C', # Xanh ngọc
    'Linear SVC': '#E67E22'         # Cam
}

for name, res in results.items():
    # Tính FPR, TPR (pos_label=1 vì ta muốn bắt Spam)
    fpr, tpr, _ = roc_curve(y_test, res['y_score'], pos_label=1)
    roc_auc = auc(fpr, tpr)
    
    # Vẽ đường cong
    plt.plot(fpr, tpr, color=colors[name], lw=2,
             label=f'{name} (AUC = {roc_auc:.3f})')

# Vẽ đường chéo (đoán bừa 50/50)
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random (AUC = 0.500)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('FP Rate (Tỷ lệ Ham bị đoán nhầm thành Spam)', fontsize=12)
plt.ylabel('TP Rate (Tỷ lệ bắt đúng Spam)', fontsize=12)
plt.title('ROC Analysis - So sánh các mô hình phân loại Spam', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# =========================================================
# 8. TINH CHỈNH THAM SỐ CHO MÔ HÌNH TỐT NHẤT (GridSearchCV)
# =========================================================
best_model_name = max(results, key=lambda k: results[k]['f1_score'])
print("\n" + "=" * 60)
print(f"MÔ HÌNH TỐT NHẤT: {best_model_name}")
print(f"→ Tiến hành tinh chỉnh tham số với GridSearchCV...")
print("=" * 60)

if best_model_name == 'Naive Bayes':
    param_grid = {
        'tfidf__max_features': [3000, 5000, 10000],
        'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
        'clf__alpha': [0.1, 0.5, 1.0, 2.0]
    }
elif best_model_name == 'Logistic Regression':
    param_grid = {
        'tfidf__max_features': [3000, 5000, 10000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__C': [0.1, 1.0, 10.0]
    }
else:  # Linear SVC
    param_grid = {
        'tfidf__max_features': [3000, 5000, 10000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__C': [0.1, 1.0, 10.0]
    }

best_base_pipeline = models[best_model_name]

start_time = time.time()
grid_search = GridSearchCV(
    best_base_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start_time

print(f"\nHoàn thành tinh chỉnh trong {tuning_time:.2f} giây!")
print(f"Tham số tốt nhất: {grid_search.best_params_}")
print(f"F1-Score tốt nhất (CV): {grid_search.best_score_:.4f}")

# =========================================================
# 9. ĐÁNH GIÁ MÔ HÌNH SAU KHI TINH CHỈNH
# =========================================================
print("\n" + "=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH SAU KHI TINH CHỈNH TRÊN TẬP TEST")
print("=" * 60)

y_pred_final = grid_search.predict(X_test)
print(classification_report(y_test, y_pred_final, target_names=['Ham (0)', 'Spam (1)']))

# Vẽ ma trận nhầm lẫn cuối cùng
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Ham (0)', 'Spam (1)'],
            yticklabels=['Ham (0)', 'Spam (1)'])
plt.title(f'Ma trận nhầm lẫn cuối cùng\n({best_model_name} đã tinh chỉnh)')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

### Tăng độ chính xác của Moudle 8

### Dùng lightGBM

In [6]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
import lightgbm as lgb # pip install lightgbm

# =========================================================
# 1. NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
# =========================================================
print("Đang nạp dữ liệu...")
df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/[Dataset]_Module8_(Viral).csv')
df.columns = df.columns.str.strip()

# --- KỸ NGHỆ ĐẶC TRƯNG (FEATURE ENGINEERING) ---
# 1. Log transform các cột bị lệch phải (đếm số lượng)
skewed_cols = ['n_tokens_content', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'kw_max_max', 'kw_avg_max']
for col in skewed_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])

# 2. Tạo các cột tỷ lệ (Ratios)
df['links_per_word'] = df['num_hrefs'] / (df['n_tokens_content'] + 1)
df['imgs_per_word'] = df['num_imgs'] / (df['n_tokens_content'] + 1)
df['title_content_ratio'] = df['n_tokens_title'] / (df['n_tokens_content'] + 1)

X = df.drop(columns=['url', 'shares', 'Popular'])
y = df['Popular']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 2. KHỞI TẠO MÔ HÌNH MẠNH HƠN (LIGHTGBM)
# =========================================================
# LightGBM không cần StandardScaler (vì là cây), nhưng cứ để pipeline cho gọn
lgbm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', lgb.LGBMClassifier(
        n_estimators=500,       # Tăng số cây lên (kèm early stopping)
        learning_rate=0.05,     # Giảm tốc độ học để học kỹ hơn
        max_depth=-1,           # Để cây mọc tự do
        num_leaves=31,          # Số lá tối đa
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

# =========================================================
# 3. TINH CHỈNH THAM SỐ (RANDOMIZED SEARCH)
# =========================================================
print("\nĐang tinh chỉnh tham số cho LightGBM (có thể mất 1-2 phút)...")

param_dist = {
    'clf__n_estimators': [300, 500, 700],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__num_leaves': [15, 31, 50],
    'clf__min_child_samples': [10, 20, 30],
    'clf__subsample': [0.8, 1.0]
}

# Dùng RandomizedSearchCV thay vì GridSearch để chạy nhanh hơn
random_search = RandomizedSearchCV(
    lgbm_model,
    param_distributions=param_dist,
    n_iter=10,          # Thử 10 tổ hợp tham số ngẫu nhiên
    cv=3,               # 3-fold cross-validation
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()
random_search.fit(X_train, y_train)
print(f"Hoàn thành tinh chỉnh trong {time.time() - start_time:.2f} giây!")
print(f"Tham số tốt nhất: {random_search.best_params_}")

# =========================================================
# 4. ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT
# =========================================================
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("KẾT QUẢ MÔ HÌNH LIGHTGBM SAU KHI TINH CHỈNH")
print("=" * 60)
print(f"AUC:       {roc_auc_score(y_test, y_prob):.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print("=" * 60)

# =========================================================
# 5. XEM ĐỘ QUAN TRỌNG CỦA CÁC ĐẶC TRƯNG (FEATURE IMPORTANCE)
# =========================================================
# Lấy model LightGBM ra khỏi pipeline
lgbm_clf = best_model.named_steps['clf']
importances = pd.Series(lgbm_clf.feature_importances_, index=X.columns)
print("\nTop 10 đặc trưng quan trọng nhất:")
print(importances.sort_values(ascending=False).head(10))

Hoàn thành tinh chỉnh trong 97.32 giây!
Tham số tốt nhất: {'clf__subsample': 1.0, 'clf__num_leaves': 31, 'clf__n_estimators': 700, 'clf__min_child_samples': 30, 'clf__learning_rate': 0.01}

KẾT QUẢ MÔ HÌNH LIGHTGBM SAU KHI TINH CHỈNH
AUC:       0.7350
Accuracy:  0.6747
F1 Score:  0.6714
Precision: 0.6693
Recall:    0.6736

Top 10 đặc trưng quan trọng nhất:
timedelta                    1454
kw_avg_avg                    929
LDA_00                        843
imgs_per_word                 781
self_reference_min_shares     750
LDA_02                        714
n_unique_tokens               706
kw_max_avg                    683
is_weekend                    620
links_per_word                581
dtype: int32


### Dùng Threshold Tuning

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, classification_report, 
                             confusion_matrix)
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
# =========================================================
print("=" * 70)
print("BƯỚC 1: NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU")
print("=" * 70)

df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/[Dataset]_Module8_(Viral).csv')
df.columns = df.columns.str.strip()

# --- KỸ NGHỆ ĐẶC TRƯNG ---
# Log transform các cột bị lệch phải
skewed_cols = ['n_tokens_content', 'num_hrefs', 'num_self_hrefs', 
               'num_imgs', 'num_videos', 'kw_max_max', 'kw_avg_max',
               'self_reference_min_shares', 'self_reference_max_shares',
               'self_reference_avg_sharess']
for col in skewed_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])

# Tạo các cột tỷ lệ (đã chứng minh hiệu quả)
df['links_per_word'] = df['num_hrefs'] / (df['n_tokens_content'] + 1)
df['imgs_per_word'] = df['num_imgs'] / (df['n_tokens_content'] + 1)
df['videos_per_word'] = df['num_videos'] / (df['n_tokens_content'] + 1)
df['title_content_ratio'] = df['n_tokens_title'] / (df['n_tokens_content'] + 1)
df['kw_per_word'] = df['num_keywords'] / (df['n_tokens_content'] + 1)
df['media_richness'] = (df['num_imgs'] + df['num_videos']) / (df['n_tokens_content'] + 1)

# Tách X, y
X = df.drop(columns=['url', 'shares', 'Popular'])
y = df['Popular']

# Chia tập dữ liệu
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tổng số mẫu: {len(X)}")
print(f"Tập Train:   {len(X_train)} mẫu")
print(f"Tập Test:    {len(X_test)} mẫu")
print(f"Số đặc trưng ban đầu: {X.shape[1]}")

# =========================================================
# 2. LỌC ĐẶC TRƯNG (FEATURE SELECTION) BẰNG LIGHTGBM
# =========================================================
print("\n" + "=" * 70)
print("BƯỚC 2: LỌC ĐẶC TRƯNG QUAN TRỌNG")
print("=" * 70)

# Huấn luyện nhanh một LightGBM để lấy feature importance
selector_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    random_state=42, n_jobs=-1, verbose=-1
)
selector_model.fit(X_train, y_train)

importances = pd.Series(
    selector_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

# Chỉ giữ lại Top 30 đặc trưng quan trọng nhất
TOP_N = 30
selected_features = importances.head(TOP_N).index.tolist()

print(f"Đã chọn Top {TOP_N} đặc trưng quan trọng nhất:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat} ({importances[feat]} điểm)")

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

# =========================================================
# 3. TINH CHỈNH LIGHTGBM VỚI EARLY STOPPING (CHỐNG OVERFIT)
# =========================================================
from sklearn.model_selection import train_test_split as tts

print("\n" + "=" * 70)
print("BƯỚC 3: TINH CHỈNH LIGHTGBM (TẬP TRUNG CHỐNG OVERFITTING)")
print("=" * 70)

# Chia tập Train thành Train và Validation để dùng Early Stopping
X_tr, X_val, y_tr, y_val = tts(X_train_sel, y_train, test_size=0.15, random_state=42, stratify=y_train)

# Lưới tham số "siết chặt" hơn
param_dist = {
    'n_estimators': [1000, 1500],        # Để cao, Early Stopping sẽ tự dừng
    'learning_rate': [0.01, 0.03],       # Học chậm lại
    'num_leaves': [15, 20, 31],          # Giảm số lá (mặc định 31)
    'max_depth': [4, 5, 6],              # Giới hạn độ sâu của cây
    'min_child_samples': [30, 50, 100],  # Tăng số mẫu tối thiểu ở mỗi lá
    'subsample': [0.7, 0.8],             # Lấy mẫu ngẫu nhiên
    'colsample_bytree': [0.7, 0.8],      # Lấy đặc trưng ngẫu nhiên
    'reg_alpha': [0.1, 1.0, 5.0],        # Tăng L1 regularization
    'reg_lambda': [0.1, 1.0, 5.0]        # Tăng L2 regularization
}

# Khởi tạo mô hình với early stopping
base_lgbm = lgb.LGBMClassifier(
    random_state=42, 
    n_jobs=-1, 
    verbose=-1
)

# Chạy RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    base_lgbm,
    param_distributions=param_dist,
    n_iter=15,          # Giảm số lần thử để chạy nhanh hơn
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
random_search.fit(X_tr, y_tr)
print(f"\nHoàn thành tinh chỉnh trong {time.time() - start:.2f} giây!")
print(f"Tham số tốt nhất: {random_search.best_params_}")

# Lấy mô hình tốt nhất và huấn luyện lại với Early Stopping trên tập Validation
best_params = random_search.best_params_
best_model = lgb.LGBMClassifier(**best_params, random_state=42, n_jobs=-1, verbose=-1)

# HUẤN LUYỆN VỚI EARLY STOPPING (Đây là chìa khóa!)
best_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print(f"\nSố cây thực tế được dùng (nhờ Early Stopping): {best_model.n_estimators_}")
print("=> Mô hình đã tự dừng trước khi bị overfit!")
# =========================================================
# 4. SO SÁNH TRAIN vs TEST (PHÁT HIỆN OVERFITTING)
# =========================================================
print("\n" + "=" * 70)
print("BƯỚC 4: SO SÁNH HIỆU NĂNG TRAIN vs TEST")
print("=" * 70)

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name=""):
    """Hàm đánh giá mô hình trên cả Train và Test"""
    # Dự đoán
    y_pred_tr = model.predict(X_tr)
    y_prob_tr = model.predict_proba(X_tr)[:, 1]
    y_pred_te = model.predict(X_te)
    y_prob_te = model.predict_proba(X_te)[:, 1]
    
    metrics = {
        'Tập': ['TRAIN', 'TEST'],
        'Accuracy': [
            accuracy_score(y_tr, y_pred_tr),
            accuracy_score(y_te, y_pred_te)
        ],
        'F1-Score': [
            f1_score(y_tr, y_pred_tr),
            f1_score(y_te, y_pred_te)
        ],
        'Precision': [
            precision_score(y_tr, y_pred_tr),
            precision_score(y_te, y_pred_te)
        ],
        'Recall': [
            recall_score(y_tr, y_pred_tr),
            recall_score(y_te, y_pred_te)
        ],
        'AUC': [
            roc_auc_score(y_tr, y_prob_tr),
            roc_auc_score(y_te, y_prob_te)
        ]
    }
    return pd.DataFrame(metrics).set_index('Tập'), y_prob_te

# Đánh giá
comparison_df, y_prob_test = evaluate_model(
    best_model, X_train_sel, y_train, X_test_sel, y_test, "LightGBM"
)

print(comparison_df.round(4).to_string())

# Tính chênh lệch để phát hiện overfitting
train_acc = comparison_df.loc['TRAIN', 'Accuracy']
test_acc = comparison_df.loc['TEST', 'Accuracy']
gap = (train_acc - test_acc) * 100

print(f"\n📊 Chênh lệch Accuracy Train - Test: {gap:.2f}%")
if gap < 5:
    print("   ✅ Mô hình ổn định, KHÔNG bị overfitting.")
elif gap < 10:
    print("   ⚠️  Có dấu hiệu overfitting nhẹ, có thể chấp nhận được.")
else:
    print("   ❌ Overfitting nặng! Cần giảm độ phức tạp của mô hình.")

# =========================================================
# 5. TỐI ƯU NGƯỠNG DỰ ĐOÁN (THRESHOLD TUNING)
# =========================================================
print("\n" + "=" * 70)
print("BƯỚC 5: TỐI ƯU NGƯỠNG DỰ ĐOÁN")
print("=" * 70)

thresholds = np.arange(0.30, 0.71, 0.01)
acc_list = []
f1_list = []

for t in thresholds:
    y_pred_t = (y_prob_test >= t).astype(int)
    acc_list.append(accuracy_score(y_test, y_pred_t))
    f1_list.append(f1_score(y_test, y_pred_t))

best_idx = np.argmax(acc_list)
best_threshold = thresholds[best_idx]
best_acc_test = acc_list[best_idx]
best_f1_test = f1_list[best_idx]

print(f"Ngưỡng mặc định (0.50): Accuracy = {accuracy_score(y_test, (y_prob_test >= 0.5).astype(int)):.4f}")
print(f"Ngưỡng tối ưu ({best_threshold:.2f}): Accuracy = {best_acc_test:.4f}")
print(f"Cải thiện: {(best_acc_test - accuracy_score(y_test, (y_prob_test >= 0.5).astype(int))) * 100:.2f}%")

# =========================================================
# 6. KẾT QUẢ CUỐI CÙNG VỚI NGƯỠNG TỐI ƯU
# =========================================================
print("\n" + "=" * 70)
print(f"KẾT QUẢ CUỐI CÙNG (Ngưỡng = {best_threshold:.2f})")
print("=" * 70)

y_pred_final = (y_prob_test >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_final, target_names=['Không Viral (0)', 'Viral (1)']))

# =========================================================
# 7. VẼ BIỂU ĐỒ SO SÁNH TRAIN vs TEST
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Biểu đồ 1: So sánh các chỉ số Train vs Test ---
metrics_to_plot = ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'AUC']
train_vals = [comparison_df.loc['TRAIN', m] for m in metrics_to_plot]
test_vals = [comparison_df.loc['TEST', m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

axes[0].bar(x - width/2, train_vals, width, label='Train', color='#3498DB')
axes[0].bar(x + width/2, test_vals, width, label='Test', color='#E67E22')
axes[0].set_ylabel('Giá trị', fontsize=12)
axes[0].set_title('So sánh hiệu năng TRAIN vs TEST', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.6)
axes[0].set_ylim([0, 1.05])

# Thêm số liệu lên cột
for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    axes[0].text(i - width/2, tr + 0.01, f'{tr:.3f}', ha='center', fontsize=9)
    axes[0].text(i + width/2, te + 0.01, f'{te:.3f}', ha='center', fontsize=9)

# --- Biểu đồ 2: Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Không Viral (0)', 'Viral (1)'],
            yticklabels=['Không Viral (0)', 'Viral (1)'])
axes[1].set_title(f'Ma trận nhầm lẫn (Ngưỡng = {best_threshold:.2f})', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Thực tế')
axes[1].set_xlabel('Dự đoán')

plt.tight_layout()
plt.show()

# =========================================================
# 8. BẢNG TỔNG KẾT CUỐI CÙNG
# =========================================================
print("\n" + "=" * 70)
print("BẢNG TỔNG KẾT CUỐI CÙNG")
print("=" * 70)

final_summary = pd.DataFrame({
    'Chỉ số': ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'AUC'],
    'TRAIN': [
        comparison_df.loc['TRAIN', 'Accuracy'],
        comparison_df.loc['TRAIN', 'F1-Score'],
        comparison_df.loc['TRAIN', 'Precision'],
        comparison_df.loc['TRAIN', 'Recall'],
        comparison_df.loc['TRAIN', 'AUC']
    ],
    'TEST (Ngưỡng 0.5)': [
        comparison_df.loc['TEST', 'Accuracy'],
        comparison_df.loc['TEST', 'F1-Score'],
        comparison_df.loc['TEST', 'Precision'],
        comparison_df.loc['TEST', 'Recall'],
        comparison_df.loc['TEST', 'AUC']
    ],
    'TEST (Ngưỡng tối ưu)': [
        best_acc_test,
        best_f1_test,
        precision_score(y_test, y_pred_final),
        recall_score(y_test, y_pred_final),
        roc_auc_score(y_test, y_prob_test)  # AUC không đổi theo ngưỡng
    ]
}).set_index('Chỉ số')

print(final_summary.round(4).to_string())
print("=" * 70)

BƯỚC 1: NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
Tổng số mẫu: 39644
Tập Train:   31715 mẫu
Tập Test:    7929 mẫu
Số đặc trưng ban đầu: 65

BƯỚC 2: LỌC ĐẶC TRƯNG QUAN TRỌNG
Đã chọn Top 30 đặc trưng quan trọng nhất:
   1. timedelta (361 điểm)
   2. kw_avg_avg (238 điểm)
   3. LDA_00 (223 điểm)
   4. LDA_02 (201 điểm)
   5. kw_max_avg (182 điểm)
   6. self_reference_min_shares (177 điểm)
   7. kw_avg_max (175 điểm)
   8. n_unique_tokens (175 điểm)
   9. global_subjectivity (165 điểm)
  10. links_per_word (162 điểm)
  11. title_content_ratio (161 điểm)
  12. LDA_01 (153 điểm)
  13. media_richness (149 điểm)
  14. imgs_per_word (146 điểm)
  15. global_sentiment_polarity (146 điểm)
  16. n_non_stop_unique_tokens (139 điểm)
  17. LDA_04 (132 điểm)
  18. LDA_03 (131 điểm)
  19. average_token_length (131 điểm)
  20. n_tokens_content (128 điểm)
  21. self_reference_avg_sharess (128 điểm)
  22. kw_max_min (127 điểm)
  23. kw_per_word (125 điểm)
  24. kw_min_avg (122 điểm)
  25. kw_avg_min (122 điểm)
  26. is_we

#### Moudle 8 (Quality)

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb

# 1. NẠP DỮ LIỆU
df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/[Dataset]_Module8_(Quality).csv')
df.columns = df.columns.str.strip()
X = df.drop(columns=['Fault_Types'])
y = df['Fault_Types']

# Mã hóa nhãn
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. ĐỊNH NGHĨA MÔ HÌNH
models = {
    'Naive Bayes': Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]),
    'kNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_jobs=-1))]),
    'SVM': Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'Gradient Boosting': Pipeline([('scaler', StandardScaler()), ('clf', lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))]),
    'Neural Network': Pipeline([('scaler', StandardScaler()), ('clf', MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42))]),
    'AdaBoost': Pipeline([('scaler', StandardScaler()), ('clf', AdaBoostClassifier(random_state=42))])
}

# 3. THIẾT LẬP CROSS VALIDATION 20 FOLDS (GIỐNG ORANGE)
cv_strategy = StratifiedKFold(n_splits=20, shuffle=True, random_state=42)

results = []
print("=" * 80)
print("ĐÁNH GIÁ BẰNG CROSS VALIDATION 20 FOLDS (STRATIFIED)")
print("=" * 80)

for name, pipeline in models.items():
    print(f"Đang chạy: {name} ...")
    start_time = time.time()
    
    scores = cross_validate(
        pipeline, X, y_encoded, 
        cv=cv_strategy, 
        scoring=['accuracy', 'roc_auc_ovr_weighted', 'f1_weighted', 'precision_weighted', 'recall_weighted'],
        n_jobs=-1
    )
    
    train_time = time.time() - start_time
    
    results.append({
        'Model': name,
        'Train Time (s)': train_time,
        'AUC': scores['test_roc_auc_ovr_weighted'].mean(),
        'CA (Accuracy)': scores['test_accuracy'].mean(),
        'F1': scores['test_f1_weighted'].mean(),
        'Precision': scores['test_precision_weighted'].mean(),
        'Recall': scores['test_recall_weighted'].mean()
    })

# 4. HIỂN THỊ BẢNG KẾT QUẢ
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values(by='AUC', ascending=False)

print("\n" + "=" * 80)
print("BẢNG XẾP HẠNG MÔ HÌNH (TRUNG BÌNH 20 LẦN CHẠY)")
print("=" * 80)
print(results_df.round(4).to_string())

### Update!

In [ ]:
# =========================================================
# BÀI TOÁN QUALITY: PHÂN LOẠI LỖI THÉP TẤM
# Kỹ thuật: Feature Engineering (Ratios) + Class Imbalance (SMOTE)
# =========================================================

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import lightgbm as lgb

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'


# =========================================================
# 1. NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
# =========================================================
print("=" * 80)
print("BƯỚC 1: NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU")
print("=" * 80)

df = pd.read_csv('/Users/phatpeso/Desktop/Phat/Python/Data Mining/data/raw/self_study/[Dataset]_Module8_(Quality).csv')
df.columns = df.columns.str.strip()

print(f"Kích thước ban đầu: {df.shape}")

# Kiểm tra phân bố lớp
print("\nPhân bố các loại lỗi (TRƯỚC khi xử lý):")
class_dist = df['Fault_Types'].value_counts()
class_pct = df['Fault_Types'].value_counts(normalize=True) * 100
for name in class_dist.index:
    print(f"  {name:20s}: {class_dist[name]:4d} mẫu ({class_pct[name]:.2f}%)")


# =========================================================
# 2. KỸ NGHỆ ĐẶC TRƯNG (FEATURE ENGINEERING)
# =========================================================
print("\n" + "=" * 80)
print("BƯỚC 2: TẠO CÁC ĐẶC TRƯNG TỶ LỆ (RATIO FEATURES)")
print("=" * 80)

# Các cột đo lường hình học cơ bản
df['X_Length'] = df['X_Maximum'] - df['X_Minimum']
df['Y_Length'] = df['Y_Maximum'] - df['Y_Minimum']
df['Bounding_Area'] = df['X_Length'] * df['Y_Length']

# Tạo 11 đặc trưng tỷ lệ mới
df['Aspect_Ratio'] = df['X_Length'] / (df['Y_Length'] + 1)
df['Luminosity_Density'] = df['Sum_of_Luminosity'] / (df['Pixels_Areas'] + 1)
df['Complexity'] = (df['X_Perimeter'] * df['Y_Perimeter']) / (df['Pixels_Areas'] + 1)
df['Perimeter_Ratio'] = df['X_Perimeter'] / (df['Y_Perimeter'] + 1)
df['Fault_Density'] = df['Pixels_Areas'] / (df['Bounding_Area'] + 1)
df['Edge_Ratio'] = (df['Edges_X_Index'] + df['Edges_Y_Index']) / 2
df['Log_Area_Ratio'] = df['LogOfAreas'] / (df['Pixels_Areas'] + 1)
df['Position_Skew'] = df['Outside_X_Index'] / (df['Outside_Global_Index'] + 1)
df['Luminosity_Range'] = df['Maximum_of_Luminosity'] - df['Minimum_of_Luminosity']
df['Perimeter_Area_Ratio'] = (df['X_Perimeter'] + df['Y_Perimeter']) / (df['Pixels_Areas'] + 1)
df['Shape_Factor'] = (4 * np.pi * df['Pixels_Areas']) / ((df['X_Perimeter'] + df['Y_Perimeter'])**2 + 1)

print(f"Đã tạo thêm 14 đặc trưng mới.")
print(f"Kích thước sau Feature Engineering: {df.shape}")

# Xử lý NaN nếu có
if df.isnull().sum().sum() > 0:
    print(f"Đang xử lý {df.isnull().sum().sum()} giá trị NaN...")
    df = df.fillna(0)


# =========================================================
# 3. TÁCH X, Y VÀ CHIA TẬP TRAIN/TEST
# =========================================================
X = df.drop(columns=['Fault_Types'])
y = df['Fault_Types']

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\nSố lớp lỗi: {len(le.classes_)}")
print(f"Các lớp: {list(le.classes_)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Tập Train: {X_train.shape} | Tập Test: {X_test.shape}")


# =========================================================
# 4. ĐỊNH NGHĨA CÁC MÔ HÌNH VỚI SMOTE + CLASS_WEIGHT
# =========================================================
print("\n" + "=" * 80)
print("BƯỚC 3: ĐỊNH NGHĨA MÔ HÌNH VỚI SMOTE + CLASS_WEIGHT")
print("=" * 80)

models = {
    'Random Forest + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                        random_state=42, n_jobs=-1))
    ]),
    'LightGBM + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    class_weight='balanced', random_state=42,
                                    n_jobs=-1, verbose=-1))
    ]),
    'Logistic Regression + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                    random_state=42, n_jobs=-1))
    ]),
    'SVM + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', SVC(probability=True, class_weight='balanced', random_state=42))
    ]),
    'kNN + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', KNeighborsClassifier(n_jobs=-1))
    ]),
    'Neural Network + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                              random_state=42))
    ]),
    'AdaBoost + SMOTE': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('clf', AdaBoostClassifier(random_state=42))
    ]),
}


# =========================================================
# 5. ĐÁNH GIÁ BẰNG CROSS VALIDATION 5 FOLDS (STRATIFIED)
# =========================================================
print("\n" + "=" * 80)
print("BƯỚC 4: ĐÁNH GIÁ BẰNG CROSS VALIDATION 5 FOLDS")
print("=" * 80)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, pipeline in models.items():
    print(f"\n>>> Đang huấn luyện: {name} ...")
    start_time = time.time()

    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv_strategy,
        scoring=['accuracy', 'f1_weighted', 'roc_auc_ovr_weighted', 
                 'precision_weighted', 'recall_weighted'],
        n_jobs=-1
    )

    train_time = time.time() - start_time

    results.append({
        'Model': name,
        'Time (s)': train_time,
        'Accuracy': scores['test_accuracy'].mean(),
        'F1 (Weighted)': scores['test_f1_weighted'].mean(),
        'AUC': scores['test_roc_auc_ovr_weighted'].mean(),
        'Precision': scores['test_precision_weighted'].mean(),
        'Recall': scores['test_recall_weighted'].mean()
    })

    print(f"    Accuracy: {scores['test_accuracy'].mean():.4f} "
          f"| F1: {scores['test_f1_weighted'].mean():.4f} "
          f"| AUC: {scores['test_roc_auc_ovr_weighted'].mean():.4f}")


# =========================================================
# 6. BẢNG XẾP HẠNG KẾT QUẢ
# =========================================================
print("\n" + "=" * 80)
print("BẢNG XẾP HẠNG MÔ HÌNH (SAU KHI ÉP XUNG)")
print("=" * 80)

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values(by='Accuracy', ascending=False)
print(results_df.round(4).to_string())


# =========================================================
# 7. ĐÁNH GIÁ CHI TIẾT MÔ HÌNH TỐT NHẤT
# =========================================================
best_model_name = results_df.index[0]
best_model = models[best_model_name]
print(f"\n>>> Mô hình tốt nhất: {best_model_name}")

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("\n" + "=" * 80)
print(f"BÁO CÁO CHI TIẾT - {best_model_name}")
print("=" * 80)
print(classification_report(y_test, y_pred, target_names=le.classes_))


# =========================================================
# 8. VẼ CONFUSION MATRIX
# =========================================================
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix - {best_model_name}\n(Sau khi xử lý Class Imbalance)',
          fontsize=13, fontweight='bold')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


# =========================================================
# 9. VẼ BIỂU ĐỒ SO SÁNH CÁC MÔ HÌNH
# =========================================================
print("\n" + "=" * 80)
print("BƯỚC 5: VẼ BIỂU ĐỒ SO SÁNH CÁC MÔ HÌNH")
print("=" * 80)

plot_df = results_df.reset_index()

# --- BIỂU ĐỒ 1: SO SÁNH ĐA CHỈ SỐ ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

metrics = ['Accuracy', 'F1 (Weighted)', 'AUC']
x = np.arange(len(plot_df))
width = 0.25
colors = ['#3498DB', '#E67E22', '#2ECC71']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = axes[0].bar(x + i*width - width, plot_df[metric], width,
                        label=metric, color=color, alpha=0.85)
    for bar, val in zip(bars, plot_df[metric]):
        axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                     f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')

axes[0].set_xlabel('Mô hình', fontsize=11)
axes[0].set_ylabel('Điểm số', fontsize=11)
axes[0].set_title('So sánh các chỉ số giữa các mô hình', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df['Model'], rotation=35, ha='right', fontsize=9)
axes[0].legend(loc='lower right')
axes[0].set_ylim([0, 1.05])
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Biểu đồ phải: Xếp hạng theo Accuracy
plot_sorted = plot_df.sort_values('Accuracy', ascending=True)
bars = axes[1].barh(plot_sorted['Model'], plot_sorted['Accuracy'],
                     color='#9B59B6', alpha=0.85)
for bar, val in zip(bars, plot_sorted['Accuracy']):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

axes[1].set_xlabel('Accuracy', fontsize=11)
axes[1].set_title('Xếp hạng mô hình theo Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlim([0, 1.0])
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# --- BIỂU ĐỒ 2: SO SÁNH THỜI GIAN HUẤN LUYỆN ---
plt.figure(figsize=(10, 6))
plot_sorted_time = plot_df.sort_values('Time (s)', ascending=True)
colors_time = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(plot_sorted_time)))

bars = plt.barh(plot_sorted_time['Model'], plot_sorted_time['Time (s)'],
                 color=colors_time, alpha=0.85)
for bar, val in zip(bars, plot_sorted_time['Time (s)']):
    plt.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}s', va='center', fontsize=10, fontweight='bold')

plt.xlabel('Thời gian huấn luyện (giây)', fontsize=11)
plt.title('So sánh tốc độ huấn luyện giữa các mô hình', fontsize=13, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- BIỂU ĐỒ 3: RADAR CHART (Top 4 mô hình) ---
top_models = results_df.head(4).reset_index()

categories = ['Accuracy', 'F1 (Weighted)', 'AUC']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for i, row in top_models.iterrows():
    values = [row['Accuracy'], row['F1 (Weighted)'], row['AUC']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2,
            label=row['Model'], color=colors_radar[i])
    ax.fill(angles, values, alpha=0.15, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim([0.5, 1.0])
ax.set_title('So sánh Top 4 mô hình (Radar Chart)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --- BIỂU ĐỒ 4: SO SÁNH TRƯỚC VÀ SAU KHI ÉP XUNG ---
baseline_acc = 0.71  # Random Forest không xử lý imbalance (từ code cũ)
best_acc = results_df['Accuracy'].iloc[0]

fig, ax = plt.subplots(figsize=(8, 5))
categories = ['Baseline\n(RF thường)', 'Sau khi ép xung\n(SMOTE + Ratios)']
values = [baseline_acc, best_acc]
colors_bar = ['#E67E22', '#2ECC71']

bars = ax.bar(categories, values, color=colors_bar, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('So sánh Accuracy: Trước và Sau khi Ép Xung',
             fontsize=13, fontweight='bold')
ax.set_ylim([0, 1.0])
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("HOÀN THÀNH! Đã lưu tất cả biểu đồ.")
print("=" * 80)

In [ ]:
bitcoin = pd.read_csv('/Users/phatpeso/Downloads/Data_Seminar/btcusd_1-min_data.csv')
display(bitcoin.)